
# Simulating Data with Leaspy

This example demonstrates how to use Leaspy to simulate longitudinal data based on a fitted model.


The following imports bring in the required modules and load the synthetic Parkinson dataset from Leaspy.
A logistic model will be fitted on this dataset and then used to simulate new longitudinal data.



In [1]:
from leaspy.datasets import load_dataset
from leaspy.io.data import Data

df = load_dataset("parkinson")

The clinical and imaging features of interest are selected and the DataFrame is converted
into a Leaspy `Data` object that can be used for model fitting.



In [2]:
data = Data.from_dataframe(
    df[
        [
            "MDS1_total",
            "MDS2_total",
            "MDS3_off_total",
            "SCOPA_total",
            "MOCA_total",
            "REM_total",
            "PUTAMEN_R",
            "PUTAMEN_L",
            "CAUDATE_R",
            "CAUDATE_L",
        ]
    ]
)

A logistic model with a two-dimensional latent space is initialized.



In [3]:
from leaspy.models import LogisticModel

model = LogisticModel(name="test-model", source_dimension=2)

The model is fitted to the data using the MCMC-SAEM algorithm.
A fixed seed is used for reproducibility and 100 iterations are performed.



In [4]:
model.fit(
    data,
    "mcmc_saem",
    n_iter=100,
    progress_bar=False,
)

/Users/jv.martini/miniforge3/envs/leaspy/lib/python3.10/site-packages/torch/__init__.py:696: UserWarning: torch.set_default_tensor_type() is deprecated as of PyTorch 2.1, please use torch.set_default_dtype() and torch.set_default_device() as alternatives. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/tensor/python_tensor.cpp:453.)
  _C._set_default_tensor_type(t)



Fit with `mcmc_saem` took: 14s


The parameters for simulating patient visits are defined.
These parameters specify the number of patients, the visit spacing, and the timing variability.



In [5]:
visit_params = {
    "patient_number": 5,
    "visit_type": "random",  # The visit type could also be 'dataframe' with df_visits.
    # "df_visits": df_test           # Example for custom visit schedule.
    "first_visit_mean": 0.0,  # The mean of the first visit age/time.
    "first_visit_std": 0.4,  # The standard deviation of the first visit age/time.
    "time_follow_up_mean": 11,  # The mean follow-up time.
    "time_follow_up_std": 0.5,  # The standard deviation of the follow-up time.
    "distance_visit_mean": 2 / 12,  # The mean spacing between visits in years.
    "distance_visit_std": 0.75
    / 12,  # The standard deviation of the spacing between visits in years.
    "min_spacing_between_visits": 1,  # The minimum allowed spacing between visits.
}

A new longitudinal dataset is simulated from the fitted model using the specified parameters.



In [6]:
df_sim = model.simulate(
    algorithm="simulate",
    features=[
        "MDS1_total",
        "MDS2_total",
        "MDS3_off_total",
        "SCOPA_total",
        "MOCA_total",
        "REM_total",
        "PUTAMEN_R",
        "PUTAMEN_L",
        "CAUDATE_R",
        "CAUDATE_L",
    ],
    visit_parameters=visit_params,
)


Simulate with `simulate` took: 0s


The simulated data is converted back to a pandas DataFrame for inspection.



In [7]:
df_sim = df_sim.data.to_dataframe()

The simulated longitudinal dataset is displayed below.



In [8]:
print(df_sim)

   ID  TIME  MDS1_total  MDS2_total  MDS3_off_total  SCOPA_total  MOCA_total  \
0   0  63.0    0.241393    0.455173        0.190662     0.090192    0.060279   
1   0  64.0    0.138206    0.134584        0.246560     0.347346    0.048247   
2   0  65.0    0.270310    0.279768        0.280613     0.211298    0.444480   
3   0  66.0    0.044993    0.212141        0.218209     0.343013    0.149863   
4   0  67.0    0.281702    0.161927        0.191817     0.249722    0.027302   
5   0  68.0    0.242691    0.266672        0.372488     0.310519    0.116539   
6   0  69.0    0.318513    0.281903        0.533288     0.374818    0.179221   
7   0  70.0    0.226605    0.287944        0.224461     0.443573    0.163519   
8   0  71.0    0.242904    0.306658        0.278724     0.254851    0.193052   
9   0  72.0    0.190786    0.411963        0.324368     0.296296    0.075056   
10  0  73.0    0.313907    0.404641        0.315539     0.315853    0.092053   
11  0  74.0    0.318274    0.350369     